# Trade and Price Book Analysis

This notebook separates the trade-tape and order-book analysis into independent cells.
The final sections add extra diagnostics for market-maker footprint, trade aggressor inference, short-horizon impact, and book replenishment.


In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
plt.style.use('ggplot')

BASE = Path('/Users/sean_tsu_/Downloads/TUTORIAL_ROUND_1')
TRADE_FILES = [
    BASE / 'trades_round_0_day_-1.csv',
    BASE / 'trades_round_0_day_-2.csv',
]
PRICE_FILES = [
    BASE / 'prices_round_0_day_-1.csv',
    BASE / 'prices_round_0_day_-2.csv',
]


def extract_day(path: Path) -> int:
    match = re.search(r'day_(-?\d+)', path.stem)
    if not match:
        raise ValueError(f'Could not parse day from {path.name}')
    return int(match.group(1))


def load_trades(files: list[Path]) -> pd.DataFrame:
    frames = []
    for path in files:
        frame = pd.read_csv(path, sep=';')
        frame['day'] = extract_day(path)
        frames.append(frame)
    trades = pd.concat(frames, ignore_index=True)
    return trades.sort_values(['symbol', 'day', 'timestamp']).reset_index(drop=True)


def load_prices(files: list[Path]) -> pd.DataFrame:
    frames = [pd.read_csv(path, sep=';') for path in files]
    prices = pd.concat(frames, ignore_index=True)
    return prices.sort_values(['product', 'day', 'timestamp']).reset_index(drop=True)


trades = load_trades(TRADE_FILES)
prices = load_prices(PRICE_FILES)
book = prices.rename(columns={'product': 'symbol'}).copy()
book['spread'] = book['ask_price_1'] - book['bid_price_1']
book['top3_bid_volume'] = book[['bid_volume_1', 'bid_volume_2', 'bid_volume_3']].fillna(0).sum(axis=1)
book['top3_ask_volume'] = book[['ask_volume_1', 'ask_volume_2', 'ask_volume_3']].fillna(0).sum(axis=1)
book['book_imbalance'] = (
    (book['top3_bid_volume'] - book['top3_ask_volume']) /
    (book['top3_bid_volume'] + book['top3_ask_volume']).replace(0, np.nan)
)
book['touch_mid'] = (book['bid_price_1'] + book['ask_price_1']) / 2
book['wall_mid'] = (
    book[['bid_price_1', 'bid_price_2', 'bid_price_3']].min(axis=1) +
    book[['ask_price_1', 'ask_price_2', 'ask_price_3']].max(axis=1)
) / 2
book['touch_vs_wall'] = book['touch_mid'] - book['wall_mid']

print('Trades shape:', trades.shape)
print('Prices shape:', prices.shape)
display(trades.head())
display(book.head())


## 1. Buyer and seller anonymity


In [ ]:
buyer_counts = trades['buyer'].value_counts(dropna=False)
seller_counts = trades['seller'].value_counts(dropna=False)

print('Buyer counts:')
display(buyer_counts)
print()
print('Seller counts:')
display(seller_counts)

anonymity_summary = pd.DataFrame({
    'missing_buyer_pct': [trades['buyer'].isna().mean()],
    'missing_seller_pct': [trades['seller'].isna().mean()],
    'trade_count': [len(trades)],
})
display(anonymity_summary)


## 2. Trade size fingerprints by product


In [ ]:
trade_size_counts = trades.groupby(['symbol', 'quantity']).size().unstack(fill_value=0)
trade_size_share = trade_size_counts.div(trade_size_counts.sum(axis=1), axis=0).round(3)

print('Trade count by product and quantity:')
display(trade_size_counts)
print()
print('Trade share by product and quantity:')
display(trade_size_share)

ax = trade_size_counts.T.plot(kind='bar', figsize=(10, 4), title='Trade size counts by product')
ax.set_xlabel('Quantity')
ax.set_ylabel('Count')
plt.show()


## 3. Time between trades


In [ ]:
trades_time = trades.copy()
trades_time['time_diff'] = trades_time.groupby(['symbol', 'day'])['timestamp'].diff()

time_diff_summary = trades_time.groupby('symbol')['time_diff'].describe()
display(time_diff_summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, symbol in zip(axes, sorted(trades_time['symbol'].unique())):
    series = trades_time.loc[trades_time['symbol'] == symbol, 'time_diff'].dropna()
    ax.hist(series, bins=30, color='#4C72B0')
    ax.set_title(f'{symbol} inter-arrival times')
    ax.set_xlabel('Timestamp gap')
plt.tight_layout()
plt.show()


## 4. Best bid / ask price footprint


In [ ]:
for symbol in sorted(book['symbol'].unique()):
    subset = book[book['symbol'] == symbol]
    print(symbol)
    print('Best bid counts:')
    display(subset['bid_price_1'].value_counts().sort_index().tail(15))
    print('Best ask counts:')
    display(subset['ask_price_1'].value_counts().sort_index().head(15))
    print('Spread distribution:')
    display(subset['spread'].value_counts().sort_index())
    print('-' * 80)


## 5. Level-1 and top-3 resting volume distributions


In [ ]:
for symbol in sorted(book['symbol'].unique()):
    subset = book[book['symbol'] == symbol]
    print(symbol)
    volume_summary = pd.DataFrame({
        'bid_volume_1': subset['bid_volume_1'].value_counts().sort_index(),
        'ask_volume_1': subset['ask_volume_1'].value_counts().sort_index(),
        'top3_bid_volume': subset['top3_bid_volume'].value_counts().sort_index(),
        'top3_ask_volume': subset['top3_ask_volume'].value_counts().sort_index(),
    }).fillna(0).astype(int)
    display(volume_summary.head(20))
    print('-' * 80)


## 6. Book imbalance and wall-mid diagnostics


In [ ]:
imbalance_summary = book.groupby('symbol')[['book_imbalance', 'touch_vs_wall']].describe().round(4)
display(imbalance_summary)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for row, symbol in enumerate(sorted(book['symbol'].unique())):
    subset = book[book['symbol'] == symbol]
    axes[row, 0].hist(subset['book_imbalance'].dropna(), bins=30, color='#55A868')
    axes[row, 0].set_title(f'{symbol} book imbalance')
    axes[row, 1].hist(subset['touch_vs_wall'].dropna(), bins=30, color='#C44E52')
    axes[row, 1].set_title(f'{symbol} touch mid - wall mid')
plt.tight_layout()
plt.show()


## 7. Trade-to-book alignment and aggressor-side inference


In [ ]:
def align_trades_to_book(trades_df: pd.DataFrame, book_df: pd.DataFrame) -> pd.DataFrame:
    aligned_parts = []
    for (symbol, day), trades_part in trades_df.groupby(['symbol', 'day'], sort=False):
        book_part = book_df[(book_df['symbol'] == symbol) & (book_df['day'] == day)].copy()
        if book_part.empty:
            continue
        merged = pd.merge_asof(
            trades_part.sort_values('timestamp'),
            book_part.sort_values('timestamp')[[
                'timestamp', 'bid_price_1', 'ask_price_1', 'bid_volume_1', 'ask_volume_1',
                'mid_price', 'touch_mid', 'spread', 'book_imbalance', 'touch_vs_wall'
            ]],
            on='timestamp',
            direction='backward'
        )
        merged['symbol'] = symbol
        merged['day'] = day
        aligned_parts.append(merged)
    return pd.concat(aligned_parts, ignore_index=True)


aligned = align_trades_to_book(trades, book)
aligned['trade_side'] = np.select(
    [aligned['price'] >= aligned['ask_price_1'], aligned['price'] <= aligned['bid_price_1']],
    ['buy_taker', 'sell_taker'],
    default='inside_or_unknown'
)
aligned['distance_to_mid'] = aligned['price'] - aligned['mid_price']

trade_side_summary = pd.crosstab(aligned['symbol'], aligned['trade_side'])
display(trade_side_summary)

display(
    aligned.groupby(['symbol', 'trade_side'])[['quantity', 'distance_to_mid', 'spread', 'book_imbalance']]
    .agg(['count', 'mean', 'median'])
    .round(3)
)


## 8. Short-horizon price impact after trades


In [ ]:
def attach_forward_book_features(trades_df: pd.DataFrame, book_df: pd.DataFrame, horizons=(1, 5, 10)) -> pd.DataFrame:
    parts = []
    for (symbol, day), trades_part in trades_df.groupby(['symbol', 'day'], sort=False):
        book_part = book_df[(book_df['symbol'] == symbol) & (book_df['day'] == day)].sort_values('timestamp').copy()
        if book_part.empty:
            continue
        features = book_part[['timestamp', 'mid_price', 'bid_price_1', 'ask_price_1', 'bid_volume_1', 'ask_volume_1']].copy()
        for horizon in horizons:
            features[f'mid_plus_{horizon}'] = book_part['mid_price'].shift(-horizon)
        merged = pd.merge_asof(
            trades_part.sort_values('timestamp'),
            features,
            on='timestamp',
            direction='backward'
        )
        merged['symbol'] = symbol
        merged['day'] = day
        parts.append(merged)
    return pd.concat(parts, ignore_index=True)


impact = attach_forward_book_features(aligned[['timestamp', 'day', 'symbol', 'price', 'quantity', 'trade_side']], book)
for horizon in (1, 5, 10):
    impact[f'impact_{horizon}'] = impact[f'mid_plus_{horizon}'] - impact['mid_price']

impact_summary = (
    impact.groupby(['symbol', 'trade_side', 'quantity'])[['impact_1', 'impact_5', 'impact_10']]
    .mean()
    .round(3)
)
display(impact_summary)

for symbol in sorted(impact['symbol'].unique()):
    subset = impact[(impact['symbol'] == symbol) & (impact['trade_side'] != 'inside_or_unknown')]
    if subset.empty:
        continue
    subset.boxplot(column='impact_5', by='trade_side', figsize=(6, 4))
    plt.title(f'{symbol} 5-step price impact by inferred trade side')
    plt.suptitle('')
    plt.xlabel('Trade side')
    plt.ylabel('Mid-price change')
    plt.show()


## 9. Simple top-of-book replenishment check


In [ ]:
def attach_next_snapshot(trades_df: pd.DataFrame, book_df: pd.DataFrame) -> pd.DataFrame:
    parts = []
    for (symbol, day), trades_part in trades_df.groupby(['symbol', 'day'], sort=False):
        book_part = book_df[(book_df['symbol'] == symbol) & (book_df['day'] == day)].sort_values('timestamp').copy()
        if book_part.empty:
            continue
        next_features = book_part[['timestamp', 'ask_price_1', 'ask_volume_1', 'bid_price_1', 'bid_volume_1']].copy()
        next_features['next_timestamp'] = book_part['timestamp'].shift(-1)
        next_features['next_ask_price_1'] = book_part['ask_price_1'].shift(-1)
        next_features['next_ask_volume_1'] = book_part['ask_volume_1'].shift(-1)
        next_features['next_bid_price_1'] = book_part['bid_price_1'].shift(-1)
        next_features['next_bid_volume_1'] = book_part['bid_volume_1'].shift(-1)
        merged = pd.merge_asof(
            trades_part.sort_values('timestamp'),
            next_features,
            on='timestamp',
            direction='backward'
        )
        merged['symbol'] = symbol
        merged['day'] = day
        parts.append(merged)
    return pd.concat(parts, ignore_index=True)


replenishment = attach_next_snapshot(aligned[['timestamp', 'day', 'symbol', 'quantity', 'trade_side']], book)
replenishment['ask_refill'] = np.where(
    replenishment['trade_side'] == 'buy_taker',
    replenishment['next_ask_volume_1'] - np.maximum(replenishment['ask_volume_1'] - replenishment['quantity'], 0),
    np.nan,
)
replenishment['bid_refill'] = np.where(
    replenishment['trade_side'] == 'sell_taker',
    replenishment['next_bid_volume_1'] - np.maximum(replenishment['bid_volume_1'] - replenishment['quantity'], 0),
    np.nan,
)

refill_summary = replenishment.groupby(['symbol', 'trade_side'])[['ask_refill', 'bid_refill']].describe().round(3)
display(refill_summary)


## 10. Prompt-style summary tables for anonymous trader personas


In [ ]:
persona_summary = (
    aligned.groupby('symbol')
    .agg(
        trades=('quantity', 'size'),
        avg_trade_size=('quantity', 'mean'),
        median_trade_size=('quantity', 'median'),
        dominant_bid=('bid_price_1', lambda s: s.mode().iat[0] if not s.mode().empty else np.nan),
        dominant_ask=('ask_price_1', lambda s: s.mode().iat[0] if not s.mode().empty else np.nan),
        dominant_spread=('spread', lambda s: s.mode().iat[0] if not s.mode().empty else np.nan),
        avg_book_imbalance=('book_imbalance', 'mean'),
    )
    .round(3)
)
display(persona_summary)

print('Use the tables above to tag recurring profiles such as:')
print('- fixed-spread market maker')
print('- fair-value step-in trader')
print('- small-lot vs large-lot taker clusters')
print('- toxic flow candidates with positive/negative post-trade impact')
